# SIH26142 — Deep Super-Resolution Fine-Tuning (Google Colab)
### Sentinel-2 ×4 Super-Resolution | Real-ESRGAN (RRDBNet Backbone)

**What this notebook does:**
1. Checks GPU, installs dependencies
2. Clones the project from GitHub and reloads updated `src` modules
3. Verifies / generates synthetic training pairs without requiring SciPy
4. Runs enhanced multi-loss fine-tuning with warmup, larger crops, SSIM tracking, and hybrid checkpoint selection
5. Plots training curves, including PSNR and SSIM
6. Runs preset-based 4× inference (`fast`, `balanced`, or `quality`) with FP16, Hann blending, TTA uncertainty, and enhanced sharpening
7. Downloads the best checkpoint and demo outputs back to this machine

> **Recommended runtime:** Runtime → Change runtime type → **T4 GPU**

## Cell 1 — GPU Check
Must show a CUDA GPU. If not, change runtime to T4 GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory / 1024**3:.1f} GB')
else:
    raise RuntimeError('No GPU detected. Go to Runtime > Change runtime type and select T4 GPU.')


## Cell 2 — Install Dependencies
Installs rasterio, scikit-image, OpenCV, Real-ESRGAN, and supporting packages. The project now avoids SciPy in pair generation for smoother Windows/Colab compatibility.

In [ ]:
# 1. Install standard scientific & image packages first
!pip install -q rasterio scikit-image tqdm opencv-python matplotlib

# 2. Create physical torchvision shim for functional_tensor
import torchvision, os
tv_dir = os.path.dirname(torchvision.__file__)
ft_path = os.path.join(tv_dir, 'transforms', 'functional_tensor.py')
with open(ft_path, 'w') as f:
    f.write('from torchvision.transforms.functional import *\n')

# 3. Install basicsr (optional, fallback provided) & realesrgan
!pip install -q --no-build-isolation basicsr || true
!pip install -q realesrgan || true
print('Dependencies ready.')


## Cell 3 — Clone Repository
Clones your GitHub repo. Uses a GitHub Personal Access Token stored in Colab Secrets for private repos.

**Setup (one-time):**
1. Go to GitHub → Settings → Developer settings → Personal access tokens → Fine-grained tokens
2. Create a token with Contents: Read permission for your Depth-Wizard repo
3. In Colab, click the 🔑 (Secrets) icon in the left sidebar
4. Add a secret named `GITHUB_TOKEN` with your token value
5. Toggle 'Notebook access' ON

In [ ]:
import os, sys

# --- Authentication for private repos ---
# Option A: Use Colab Secrets (recommended)
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/AjayBora002/Depth-Wizard.git'
    print('Using GitHub token from Colab Secrets')
except (ImportError, userdata.SecretNotFoundError):
    # Option B: Public repo fallback (no token needed)
    REPO_URL = 'https://github.com/AjayBora002/Depth-Wizard.git'
    print('No GITHUB_TOKEN found in Colab Secrets — using public URL')

CLONE_DIR = '/content/Depth-Wizard'
PROJECT_DIR = f'{CLONE_DIR}/srm-project'

if not os.path.exists(CLONE_DIR):
    !git clone {REPO_URL} {CLONE_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {CLONE_DIR} pull

# Clear previously imported src modules so new git pull changes take effect
for mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[mod]

# Add project root to path so 'src' is importable
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
!ls src/


## Cell 4 — Download Pretrained Weights
Auto-downloads `RealESRGAN_x4plus.pth` into `src/checkpoints/` if not already present.

In [ ]:
from src.model import _download_weights
weight_path = _download_weights('x4plus')
print('Weights ready at:', weight_path)


## Cell 5 — Generate Synthetic Training Pairs
If pairs already exist in `data/synthetic_pairs/`, this step is skipped.
Otherwise it synthesises LR/HR pairs from the Sentinel-2 tiles in `data/raw/`.

> **Note:** The repo should already contain sample `.npy` pairs committed to `data/synthetic_pairs/`.
> If not, you need to either upload tiles to `data/raw/` and run generation,
> or upload the pairs directly.

In [ ]:
import os
from pathlib import Path

lr_dir = Path('data/synthetic_pairs/lr')
hr_dir = Path('data/synthetic_pairs/hr')
n_lr = len(list(lr_dir.glob('*.npy'))) if lr_dir.exists() else 0
n_hr = len(list(hr_dir.glob('*.npy'))) if hr_dir.exists() else 0
print(f'Found {n_lr} LR pairs and {n_hr} HR pairs')

if n_lr == 0 or n_hr == 0:
    raw_tiles = list(Path('data/raw').glob('*.tif'))
    if not raw_tiles:
        print('WARNING: No tiles in data/raw/ and no pairs found.')
        print('Upload .tif tiles to data/raw/ via Files panel, then re-run this cell.')
    else:
        print(f'Generating pairs from {len(raw_tiles)} tile(s)...')
        from src.pair_generation import generate_all_pairs
        generate_all_pairs(
            raw_dir='data/raw',
            out_dir='data/synthetic_pairs',
            scale=4,
            patches_per_tile=50,
        )
        print('Pair generation complete.')
else:
    print(f'Pairs already present ({n_lr} LR / {n_hr} HR). Skipping generation.')


## Cell 6 — Enhanced Fine-Tune the Model
Runs enhanced multi-loss training: **L1 + Perceptual (VGG16) + SAM + Edge + FFT**, with warmup + cosine LR scheduling, SSIM tracking, and hybrid **PSNR + SSIM** checkpoint selection.

**Memory-safe defaults for Colab T4 (~15 GB VRAM):**
- `batch_size=2` (reduced from 4)
- `crop_size=128` (reduced from 192)
- `gradient_checkpointing=True` (saves ~40-60% VRAM)

50 epochs ≈ 30–90 minutes on T4.

The `time_budget_hours=2` cap is a safety net — adjust or remove as needed.

In [ ]:
# Clear cached src modules so updated files from git pull are reloaded
import sys, torch
for mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[mod]

# Free any leftover GPU memory from previous cells
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB allocated, '
          f'{torch.cuda.memory_reserved()/1024**3:.2f} GB reserved')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

# Enhanced SIH training recipe:
# - Gradient checkpointing saves ~40-60% VRAM (critical for T4)
# - batch_size=2 + crop_size=128 fits within 15GB T4 VRAM
# - warmup + cosine LR schedule
# - SSIM metric tracking
# - best checkpoint selected by PSNR + SSIM hybrid score
from src.train_enhanced import train_enhanced

results = train_enhanced(
    pairs_dir='data/synthetic_pairs',
    output_dir='src/checkpoints',
    model_key='x4plus',
    epochs=50,
    batch_size=2,              # T4-safe (was 4 — caused OOM)
    crop_size=128,             # T4-safe (was 192 — caused OOM)
    lr=5e-5,
    warmup_epochs=5,
    lambda_perceptual=0.15,
    lambda_sam=0.03,
    lambda_edge=0.08,
    lambda_freq=0.03,
    save_every=5,
    num_workers=2,
    use_amp=True,              # AMP (FP16) for T4 speed boost
    gradient_checkpointing=True,  # Saves ~40-60% VRAM
    time_budget_hours=2.0,     # Stop cleanly if Colab session nears timeout
)

print('\nEnhanced training complete!')
print('Best checkpoint:', results['best_checkpoint'])
hist = results['history']
print(f'Epochs run: {len(hist["train_loss"])}')
print(f'Best Val PSNR: {max(hist["val_psnr"]):.2f} dB')
if 'val_ssim' in hist:
    print(f'Best Val SSIM: {max(hist["val_ssim"]):.4f}')
print(f'Total training time: {sum(hist["epoch_time"])/60:.1f} min')

## Cell 7 — Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt

with open('src/checkpoints/training_history.json') as f:
    hist = json.load(f)

epochs = range(1, len(hist['train_loss']) + 1)
has_ssim = 'val_ssim' in hist and len(hist['val_ssim']) == len(hist['train_loss'])

ncols = 4 if has_ssim else 3
fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4))

axes[0].plot(epochs, hist['train_loss'], 'b-o', markersize=4, label='Train')
axes[0].plot(epochs, hist['val_loss'], 'r-o', markersize=4, label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, hist['val_psnr'], 'g-o', markersize=4)
axes[1].set_title('Validation PSNR (dB)')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

metric_axis = 2
if has_ssim:
    axes[2].plot(epochs, hist['val_ssim'], 'c-o', markersize=4)
    axes[2].set_title('Validation SSIM')
    axes[2].set_xlabel('Epoch')
    axes[2].grid(True, alpha=0.3)
    metric_axis = 3

axes[metric_axis].plot(epochs, hist['epoch_time'], 'm-o', markersize=4)
axes[metric_axis].set_title('Epoch Time (s)')
axes[metric_axis].set_xlabel('Epoch')
axes[metric_axis].grid(True, alpha=0.3)

plt.suptitle('SIH26142 — Enhanced SR Fine-Tuning Progress', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('src/checkpoints/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Curves saved to src/checkpoints/training_curves.png')

## Cell 8 — Run 4× Super-Resolution Inference (Preset-Based, GPU Accelerated)
Runs full pipeline inference on Sentinel-2 tiles with **FP16 (`--half`)**, **Hann-window blending**, optional **batched TTA uncertainty**, and enhanced post-processing.

Choose one preset:
- `fast`: largest patches, uncertainty off, best for quick demos
- `balanced`: optimized default, 4× TTA uncertainty
- `quality`: smaller patches, 8× TTA, stronger sharpening / high-quality settings

On a T4 GPU, the 512×512 demo tile is intended to stay Colab-friendly.

In [ ]:
import os, sys
from pathlib import Path

# Pick an inference preset: 'fast', 'balanced', or 'quality'
INFERENCE_PRESET = 'balanced'

# 1. Ensure latest files and satellite tiles are present
raw_tiles = list(Path('data/raw').glob('*.tif'))
if not raw_tiles:
    print('data/raw/ empty, pulling latest files from GitHub...')
    !git pull
    raw_tiles = list(Path('data/raw').glob('*.tif'))

if not raw_tiles:
    raise FileNotFoundError('No .tif tiles found in data/raw/. Run Cell 3 first.')

# Prefer the 512x512 tile for fast demo inference (avoids Colab RAM crashes on large scenes)
sample_512 = Path('data/raw/S2_Delhi_Sample_512.tif')
input_tile = str(sample_512 if sample_512.exists() else raw_tiles[0])
print(f'Input tile: {input_tile}')

# 2. Find best checkpoint
ckpt_path = 'src/checkpoints/model_finetuned_best.pth'
if not os.path.exists(ckpt_path):
    ckpts = list(Path('src/checkpoints').glob('*.pth'))
    ckpt_path = str(ckpts[0]) if ckpts else None
print(f'Using checkpoint: {ckpt_path}')

# 3. Clear module cache and run inference directly
for mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[mod]

from src.inference import run_inference

PRESETS = {
    'fast':     dict(patch_size=512, overlap=48, compute_uncertainty=False, tta_n=0, sharpen=0.4, high_quality=False),
    'balanced': dict(patch_size=512, overlap=64, compute_uncertainty=True,  tta_n=4, sharpen=0.5, high_quality=False),
    'quality':  dict(patch_size=384, overlap=64, compute_uncertainty=True,  tta_n=8, sharpen=0.6, high_quality=True),
}
settings = PRESETS[INFERENCE_PRESET]
print(f'Inference preset: {INFERENCE_PRESET} -> {settings}')

output_file = f'data/outputs/sr_delhi_512_{INFERENCE_PRESET}_finetuned.tif'
result = run_inference(
    input_path=input_tile,
    output_path=output_file,
    checkpoint_path=ckpt_path,
    half=True,
    tile_size=256,
    **settings,
)

print('\n' + '=' * 50)
print(f'SR GeoTIFF saved: {result["sr_output"]}')
print(f'Uncertainty map: {result["uncertainty_map"]}')
print('=' * 50)

## Cell 9 — High-Resolution Visualizer & Uncertainty Map
Displays side-by-side comparison: Input Tile vs 4× Super-Resolved vs TTA Uncertainty.

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

try:
    INFERENCE_PRESET
except NameError:
    INFERENCE_PRESET = 'balanced'

def norm_rgb(arr):
    rgb = arr[:3].transpose(1, 2, 0).astype(np.float32)
    p2, p98 = np.percentile(rgb, (2, 98))
    if p98 > p2:
        rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)
    else:
        rgb = np.clip(rgb / 255.0, 0, 1)
    return rgb

lr_file = Path('data/raw/S2_Delhi_Sample_512.tif')
if not lr_file.exists():
    raw_tiles = list(Path('data/raw').glob('*.tif'))
    if not raw_tiles:
        raise FileNotFoundError('No input tile found in data/raw/. Run Cell 8 first!')
    lr_file = raw_tiles[0]

with rasterio.open(lr_file) as f_lr:
    lr_data = f_lr.read()
    lr_rgb = norm_rgb(lr_data)

sr_file = Path(f'data/outputs/sr_delhi_512_{INFERENCE_PRESET}_finetuned.tif')
if not sr_file.exists():
    alts = sorted(Path('data/outputs').glob('*_sr_x*.tif')) + sorted(Path('data/outputs').glob('sr_delhi_512_*_finetuned.tif'))
    if alts:
        sr_file = alts[0]
    else:
        raise FileNotFoundError('No SR output found in data/outputs/. Run Cell 8 first!')

with rasterio.open(sr_file) as f_sr:
    sr_data = f_sr.read()
    sr_rgb = norm_rgb(sr_data)

stem = sr_file.stem
unc_path = sr_file.with_name(f'{stem}_uncertainty.npy')
has_unc = unc_path.exists()
if has_unc:
    unc = np.load(unc_path)
    unc_map = np.mean(unc, axis=0) if unc.ndim == 3 else unc

# Crop coordinates around town/field features in center
r0, r1 = int(lr_rgb.shape[0] * 0.42), int(lr_rgb.shape[0] * 0.54)
c0, c1 = int(lr_rgb.shape[1] * 0.42), int(lr_rgb.shape[1] * 0.54)
lr_crop = lr_rgb[r0:r1, c0:c1]
scale_y = sr_rgb.shape[0] / lr_rgb.shape[0]
scale_x = sr_rgb.shape[1] / lr_rgb.shape[1]
sr_crop = sr_rgb[int(r0*scale_y):int(r1*scale_y), int(c0*scale_x):int(c1*scale_x)]

import cv2
bicubic_crop = cv2.resize(lr_crop, (sr_crop.shape[1], sr_crop.shape[0]), interpolation=cv2.INTER_CUBIC)

# Figure with 2 rows: Row 1 = Full Scene, Row 2 = Zoomed-in 4x Detail
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.2])

ax0 = fig.add_subplot(gs[0, 0])
ax0.imshow(lr_rgb)
ax0.set_title(f'Full Scene: Low-Res Input ({lr_rgb.shape[1]}x{lr_rgb.shape[0]})', fontsize=12)
ax0.axis('off')

ax1 = fig.add_subplot(gs[0, 1])
ax1.imshow(sr_rgb)
ax1.set_title(f'Full Scene: 4x SR — {INFERENCE_PRESET.title()} ({sr_rgb.shape[1]}x{sr_rgb.shape[0]})', fontsize=12, fontweight='bold', color='green')
ax1.axis('off')

ax2 = fig.add_subplot(gs[0, 2])
if has_unc:
    im = ax2.imshow(unc_map, cmap='viridis')
    ax2.set_title('TTA Uncertainty Map (Confidence)', fontsize=12)
    plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
else:
    ax2.text(0.5, 0.5, 'Uncertainty disabled\nfor this preset', ha='center', va='center', fontsize=13)
    ax2.set_title('TTA Uncertainty Map', fontsize=12)
ax2.axis('off')

# Row 2: Zoomed-In Details
ax3 = fig.add_subplot(gs[1, 0])
ax3.imshow(lr_crop, interpolation='nearest')
ax3.set_title('Zoomed-In: Original 10m Pixels (Blocky)', fontsize=13, fontweight='bold', color='red')
ax3.axis('off')

ax4 = fig.add_subplot(gs[1, 1])
ax4.imshow(bicubic_crop)
ax4.set_title('Zoomed-In: Standard Bicubic 4x (Blurry)', fontsize=13, fontweight='bold', color='orange')
ax4.axis('off')

ax5 = fig.add_subplot(gs[1, 2])
ax5.imshow(sr_crop)
ax5.set_title('Zoomed-In: 4x AI Super-Resolved (Crisp & Sharp)', fontsize=13, fontweight='bold', color='green')
ax5.axis('off')

plt.suptitle('SIH26142 Sentinel-2 Super-Resolution: Full Scene & Pixel-Level Detail', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('data/outputs/sr_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

## Cell 10 — Launch Interactive Streamlit Dashboard (GPU Powered)
Runs the SRM Web App inside Colab with a secure public tunnel (`localtunnel`).

1. Run the cell below.
2. Copy the **Tunnel Password (IP)** displayed.
3. Click the **loca.lt URL**, paste the IP into the box, and click Submit!

In [ ]:
!pip install -q streamlit streamlit-folium streamlit-image-comparison pydeck plotly
!npm install -g localtunnel -q

import urllib
public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print('=' * 60)
print(f'YOUR TUNNEL PASSWORD (IP): {public_ip}')
print('=' * 60)

!streamlit run dashboard/app.py --server.port 8501 --server.headless true & npx localtunnel --port 8501


## Cell 11 — Download Outputs & Checkpoint
Downloads your fine-tuned model checkpoint and super-resolved GeoTIFF files.

In [ ]:
from google.colab import files
import os
from pathlib import Path

try:
    INFERENCE_PRESET
except NameError:
    INFERENCE_PRESET = 'balanced'

candidate_outputs = [
    f'data/outputs/sr_delhi_512_{INFERENCE_PRESET}_finetuned.tif',
    f'data/outputs/sr_delhi_512_{INFERENCE_PRESET}_finetuned_uncertainty.npy',
    'data/outputs/sr_comparison.png',
]

# Include any fallback SR outputs generated by src/inference.py naming.
candidate_outputs.extend(str(p) for p in sorted(Path('data/outputs').glob('*_sr_x*.tif')))
candidate_outputs.extend(str(p) for p in sorted(Path('data/outputs').glob('*_uncertainty.npy')))

to_download = [
    'src/checkpoints/model_finetuned_best.pth',
    'src/checkpoints/model_finetuned_final.pth',
    'src/checkpoints/training_history.json',
    'src/checkpoints/training_curves.png',
    *candidate_outputs,
]

seen = set()
for path in to_download:
    if path in seen:
        continue
    seen.add(path)
    if os.path.exists(path):
        print(f'Downloading {path}...')
        files.download(path)
    else:
        print(f'SKIP (not found): {path}')

print('Done!')